In [ ]:
from google.cloud import bigquery
from pathlib import Path
import os
import requests
import pandas as pd
import db_dtypes
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from hospital_readmission_risk import config, data, preprocessing, models

import importlib

In [ ]:
def estimate_cost_reduction(df_cost, pred_values, r = 0.2):

    gains = pd.DataFrame(index = df_cost.index)

    for col in df_cost.columns:

        if ('d30' in col):

            pos = col.rfind("_")
            model = col[:pos]
            threshold = float(col[(pos + 1):])

            for id in df_cost.index:

                if(df_cost.loc[id, col] == 1):

                    exp_avoided_cost = r * pred_values.loc[id, model] * df_cost.loc[id, 'total_readmission_cost']

                    relative_prob = pred_values.loc[id, model] / threshold

                    if(relative_prob < 2 * threshold):

                        intervention_cost = min(df_cost.loc[id, 'cost_per_day_stay'], df_cost.loc[id, 'avg_cost_of_prev_stays'])

                    elif(relative_prob < 3 * threshold):

                        intervention_cost = 2 * min(df_cost.loc[id, 'cost_per_day_stay'], df_cost.loc[id, 'avg_cost_of_prev_stays'])
                    
                    else:

                        intervention_cost = 3 * min(df_cost.loc[id, 'cost_per_day_stay'], df_cost.loc[id, 'avg_cost_of_prev_stays'])

                    gains.loc[id, col] = exp_avoided_cost - intervention_cost

                else: 

                    gains.loc[id, col] = 0

    gains.loc['total_avoided'] = gains.sum(axis = 0)

    return gains

In [ ]:
def build_models(models, df_numeric, df_results):

    test_log = pd.DataFrame(columns = ['roc_auc', 'pr_auc'])

    metrics_log = pd.DataFrame(columns = ['roc', 'pr', 'brier_loss_total', 'brier_loss_half', 'precision', 'recall', 'f1'])

    coefs = pd.DataFrame(index = df_numeric.columns)

    pred_values = pd.DataFrame()

    for model_name in models:

        for set in config.MODEL_CONFIGS:

            if set['name'] == model_name:

                params = set['params']

        X_train, X_test, y_train, y_test = make_train_test_split(df_numeric, df_results['readmit_30d'])
        
        trained_pipe, name, test_log = train_model(X_train, y_train, model_name, models[model_name], params, test_log)

        if pred_values.shape[0] < 2:

            pred_values['X'] = X_test.index

            pred_values.set_index('X', inplace = True)

        if pred_values.shape[1] < 1:

            pred_values['readmit_30d'] = y_test

        coefs, metrics_log, pred_values = evaluate_model(X_test, y_test, name, trained_pipe, coefs, metrics_log, pred_values)

        X_train, X_test, y_train, y_test = make_train_test_split(df_numeric, df_results['readmit_90d'])
        
        trained_pipe, name, test_log = train_model(X_train, y_train, model_name, models[model_name], params, test_log, d30 = False)

        if pred_values.shape[1] < 3:

            pred_values['readmit_90d'] = y_test

        coefs, metrics_log, pred_values = evaluate_model(X_test, y_test, name, trained_pipe, coefs, metrics_log, pred_values)

        thresholds, threshold_metrics = build_threshold_metrics(pred_values)

    df_cost = cost_reduction_preprocessor(df_raw, thresholds, df_results)

    gains = estimate_cost_reduction(df_cost, pred_values)

    positive_model_mask = gains.loc['total_avoided'] > 0

    gains_positive = gains.loc[:, positive_model_mask]

    rows = pred_values[pred_values['readmit_30d'] == 1].index

    gains_positive.loc['percent_cost_saved'] = gains_positive.loc['total_avoided'] / df_raw.loc[rows, 'total_readmission_cost'].sum()

    dict = {
        'test_log': test_log,
        'coefs': coefs,
        'metrics_log': metrics_log,
        'pred_values': pred_values,
        'thresholds': thresholds,
        'threshold_metrics': threshold_metrics,
        'df_cost': df_cost,
        'gains': gains,
        'gains_positive': gains_positive,
    }

    return dict

In [ ]:
df_raw = data.load_data(config.data_path)
df_numeric, df_results = preprocessing.preprocess_train_data(df_raw)

results = models.build_models(config.models, df_numeric, df_results)

results['thresholds'], results['threshold_metrics'] = build_threshold_metrics(results['pred_values'])

C:\Users\4infi\AppData\Local\Temp\ipykernel_23420\2591555852.py:6: DtypeWarning: Columns (37) have mixed types. Specify dtype option on import or set low_memory=False.
  return pd.read_csv(path, sep = ',')
C:\Users\4infi\AppData\Local\Temp\ipykernel_23420\2235083803.py:14: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_numeric['avg_cost_of_prev_stays'] = df_numeric['avg_cost_of_prev_stays'].fillna(0)
C:\Users\4infi\AppData\Local\Temp\ipykernel_23420\2235083803.py:15: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a

[LightGBM] [Info] Number of positive: 4119, number of negative: 83483
[LightGBM] [Info] Total Bins 1430
[LightGBM] [Info] Number of data points in the train set: 87602, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.047019 -> initscore=-3.009033
[LightGBM] [Info] Start training from score -3.009033


d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


[LightGBM] [Info] Number of positive: 8044, number of negative: 79558
[LightGBM] [Info] Total Bins 1430
[LightGBM] [Info] Number of data points in the train set: 87602, number of used features: 21
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.091824 -> initscore=-2.291560
[LightGBM] [Info] Start training from score -2.291560


d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
d:\Python Projects\Hospital readmission risk\.venv\Lib\site-packages\sklearn\utils\validation.py:2691: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
C:\Users\4infi\AppData\Local\Temp\ipykernel_23420\1642686626.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  thresholds[model + '_' + str(t)] = (values[model] >= t).astype(int)
C:\Users\4infi\AppData\Local\Temp\ipykernel_23420\1642686626.py:17: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result

In [ ]:
model_data['gains_positive']
    

,logreg_d30_0.7,logreg_d30_0.75,logreg_d30_0.8,logreg_d30_0.85,logreg_d30_0.9,logreg_d30_0.95,rf_d30_0.75,rf_d30_0.8,rf_d30_0.85,rf_d30_0.9
X,,,,,,,,,,
108992,-28999.985863,-6.527786e+03,-6.527786e+03,-6.527786e+03,-6.527786e+03,-6.527786e+03,-10026.540528,0.000000,0.000000,0.000000
52054,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000
466,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000
6985,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000
27343,-2074.070000,-2.074070e+03,-2.074070e+03,-2.074070e+03,-2.074070e+03,-2.074070e+03,0.000000,0.000000,0.000000,0.000000
...,...,...,...,...,...,...,...,...,...,...
16716,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000
41507,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000
72181,0.000000,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000e+00,0.000000,0.000000,0.000000,0.000000
